In [ ]:
%pip install uv -q

In [ ]:
!uv init --bare

In [ ]:
!apt-get -qq -y install espeak-ng > /dev/null 2>&1

In [ ]:
!uv pip install --system -q piper-sample-generator soundfile librosa matplotlib pandas

In [ ]:
import os
import sys

REPO_DIR = os.path.abspath("piper-sample-generator")
if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 -q https://github.com/rhasspy/piper-sample-generator.git {REPO_DIR}

# import piper_sample_generator and piper_train from the clone, not from site-packages
sys.path.insert(0, REPO_DIR)

# only the weights are a release asset; the .pt.json config comes with the clone
MODEL_PATH = f"{REPO_DIR}/models/en_US-libritts_r-medium.pt"
if not os.path.exists(MODEL_PATH):
    !wget -q -O {MODEL_PATH} 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'

!ls -lh {REPO_DIR}/models/en_US-libritts_r-medium.pt*

In [ ]:
import itertools as it
import json
import logging

from piper_sample_generator.__main__ import generate_samples
from IPython.display import display, Audio
import soundfile as sf
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import librosa
import librosa.display

# the package sets root logging to DEBUG on import
logging.getLogger().setLevel(logging.WARNING)

In [ ]:
with open(f"{MODEL_PATH}.json") as f:
    config = json.load(f)

NUM_SPEAKERS = config["num_speakers"]
SAMPLE_RATE = config["audio"]["sample_rate"]
print(f"{NUM_SPEAKERS} speakers @ {SAMPLE_RATE} Hz")

In [ ]:
OUTPUT_DIR = "piper_exploration"
os.makedirs(f"{OUTPUT_DIR}/positive", exist_ok=True)
os.makedirs(f"{OUTPUT_DIR}/negative", exist_ok=True)

In [ ]:
WAKE_WORD = "hey sonny"

generate_samples(
    text=WAKE_WORD,
    output_dir=f"{OUTPUT_DIR}/test",
    model=MODEL_PATH,
    max_samples=1,
    batch_size=1,
)

data, sr = sf.read(f"{OUTPUT_DIR}/test/0.wav")
display(Audio(data=data, rate=sr, autoplay=True))
print(f"Duration: {len(data)/sr:.2f}s")

### How Piper picks speakers

There is no named voice list. `generate_samples` walks
`itertools.product(range(num_speakers), range(num_speakers))` and SLERP-blends each
**pair** of speaker embeddings, so every clip is a mix of two LibriTTS-R speakers.

The catch: the product is ordered, so with all 904 speakers the first 904 samples are
all `(0, k)` — speaker 0 is in every single clip. Small exploration runs get far less
diversity than the speaker count suggests.

Fix: cap `max_speakers` so the pair space is small enough to traverse. With
`max_speakers = n` there are `n**2` pairs, so pick `n` near `sqrt(max_samples)`.
Upstream also recommends staying under 904 because the rarest speakers produce artifacts.

In [ ]:
MAX_SPEAKERS = 10          # 10 x 10 = 100 distinct speaker pairs
MAX_SAMPLES = 100          # exactly one pass over the pair space
BATCH_SIZE = 1             # settings rotate per batch; raise on GPU for scale-up

# note: length_scale below 1 is faster, above 1 is slower
LENGTH_SCALES = [0.85, 1.0, 1.15]
SLERP_WEIGHTS = [0.5]
NOISE_SCALES = [0.667]
NOISE_SCALE_WS = [0.8]

print(f"{MAX_SAMPLES} samples over {MAX_SPEAKERS**2} speaker pairs x {len(LENGTH_SCALES)} length scales")

### Manifest

Use deterministic iteration order to replay it to recover which speak pair settings produced each clip.

In [ ]:
def build_manifest(output_dir, texts, max_samples, batch_size, num_speakers,
                   slerp_weights, length_scales, noise_scales, noise_scale_ws):
    speakers = it.cycle(it.product(range(num_speakers), range(num_speakers)))
    settings = it.cycle(it.product(slerp_weights, length_scales, noise_scales, noise_scale_ws))
    text_iter = it.cycle(texts)

    records = []
    for idx in range(max_samples):
        if idx % batch_size == 0:
            slerp_weight, length_scale, noise_scale, noise_scale_w = next(settings)
        speaker_1, speaker_2 = next(speakers)
        path = f"{output_dir}/{idx}.wav"
        data, sr = sf.read(path)
        records.append({
            "path": path, "text": next(text_iter),
            "speaker_1": speaker_1, "speaker_2": speaker_2,
            "slerp_weight": slerp_weight, "length_scale": length_scale,
            "noise_scale": noise_scale, "noise_scale_w": noise_scale_w,
            "duration_s": len(data) / sr,
        })
    return pd.DataFrame(records)

Generating positives batch

In [ ]:
generate_samples(
    text=WAKE_WORD,
    output_dir=f"{OUTPUT_DIR}/positive",
    model=MODEL_PATH,
    max_samples=MAX_SAMPLES,
    batch_size=BATCH_SIZE,
    max_speakers=MAX_SPEAKERS,
    slerp_weights=SLERP_WEIGHTS,
    length_scales=LENGTH_SCALES,
    noise_scales=NOISE_SCALES,
    noise_scale_ws=NOISE_SCALE_WS,
)

df = build_manifest(
    f"{OUTPUT_DIR}/positive", [WAKE_WORD], MAX_SAMPLES, BATCH_SIZE, MAX_SPEAKERS,
    SLERP_WEIGHTS, LENGTH_SCALES, NOISE_SCALES, NOISE_SCALE_WS,
)
df.describe()

Sanity check of generated samples on normal speed

In [ ]:
def listen_to_generated_samples(df: pd.DataFrame) -> None:
    for row in df.itertuples():
        print(f"Speakers: {row.speaker_1}+{row.speaker_2}; Length scale: {row.length_scale}; "
              f"Duration: {row.duration_s:.2f}s; Text: {row.text}")
        data, sr = sf.read(row.path)
        display(Audio(data=data, rate=sr))

In [ ]:
listen_to_generated_samples(df[df.length_scale == 1.0])

Duration is the cheapest outlier signal — a clip far from the median usually means a
dropped or smeared phrase.

In [ ]:
median = df.duration_s.median()
outliers = df[(df.duration_s < 0.6 * median) | (df.duration_s > 1.6 * median)]
print(f"median {median:.2f}s | {len(outliers)} outliers")
listen_to_generated_samples(outliers)

Visual Inspection: waveform + spectogram

In [ ]:
items_path = df.sample(5)["path"].tolist()

fig, axes = plt.subplots(len(items_path), 2, figsize=(10, 3 * len(items_path)))
for row_ax, path in zip(axes, items_path):
    y, sr = librosa.load(path, sr=None)
    row_ax[0].plot(y)
    row_ax[0].set_title(f"Waveform: {os.path.basename(path)}")
    S = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    librosa.display.specshow(S, sr=sr, x_axis="time", y_axis="hz", ax=row_ax[1])
    row_ax[1].set_title("Spectrogram")
plt.tight_layout()
plt.show()

### Testing negative samples

Same speaker pool as the positives on purpose — if negatives came from a different set of
voices the model could learn voice identity instead of the phrase.

`hey sunny` is deliberately absent: it is a homophone of the target and cannot be
separated acoustically, so labelling it negative would fight the positive set.

In [ ]:
NEGATIVE_PHRASES = [
    "hello", "good morning", "what time is it",
    "play some music", "turn off the lights",
    "hey siri", "okay google", "alexa",
    # phonetically close to "hey sonny"
    "sonny", "hey honey", "hey johnny", "hi sonny", "okay sonny",
]

MAX_NEGATIVE_SAMPLES = len(NEGATIVE_PHRASES) * 4

generate_samples(
    text=NEGATIVE_PHRASES,
    output_dir=f"{OUTPUT_DIR}/negative",
    model=MODEL_PATH,
    max_samples=MAX_NEGATIVE_SAMPLES,
    batch_size=BATCH_SIZE,
    max_speakers=MAX_SPEAKERS,
    slerp_weights=SLERP_WEIGHTS,
    length_scales=LENGTH_SCALES,
    noise_scales=NOISE_SCALES,
    noise_scale_ws=NOISE_SCALE_WS,
)

neg_df = build_manifest(
    f"{OUTPUT_DIR}/negative", NEGATIVE_PHRASES, MAX_NEGATIVE_SAMPLES, BATCH_SIZE, MAX_SPEAKERS,
    SLERP_WEIGHTS, LENGTH_SCALES, NOISE_SCALES, NOISE_SCALE_WS,
)
neg_df.describe()

In [ ]:
listen_to_generated_samples(neg_df.sample(10))

## Augmentation

Built-in augmentation handles following:

1. Randomly decrease the volume
2. Change the acoustics of the sample to sound like the speaker was in a room with echo or using a poor quality microphone
3. Resample to 16Khz for training openWakeWord

In [ ]:
!PYTHONPATH={REPO_DIR} python3 -m piper_sample_generator.augment --sample-rate 16000 {OUTPUT_DIR}/positive {OUTPUT_DIR}/positive_16k

aug_path = f"{OUTPUT_DIR}/positive_16k/0.wav"
data, sr = sf.read(aug_path)
print(f"{aug_path}: {sr} Hz, {len(data)/sr:.2f}s")
display(Audio(data=data, rate=sr))